# RT channel (8B) — Approach A: RT as binned tokens

We proved scale doesn't help (70B == 8B). The only way to break the thin-residual
ceiling is to give the model the **reaction-time** channel it never had.

**Key (your) insight:** Minitaur was trained choice-only, so it cannot predict RT.
We must FIRST train a population model that learns RT (**M_pop_RT**); only then can
the individual signal ride on the RT residual.

**Approach A** (this notebook): render a per-trial **log-RT decile token** inside
`<<>>` (e.g. `You press <<larger_later>>. <<rt7>>`). Because it's inside `<<>>` it is
loss-bearing, so the *entire existing pipeline* (masking / M_pop / surprise) works
unchanged — the model now predicts choice **and** RT.

Pipeline: **train M_pop_RT → surprise transfer on RT data → compare within-id to the
choice-only baseline (~0.14).** If it jumps, RT carries the missing individual signal.

## Get the RT data onto Drive (one-time)

The RT-rendered transcripts were built locally with `centaur_render.py --with-rt`
(choice + log-RT decile per trial), 11 starting-subset tasks, complete + retest.
**Upload the local `output_nl_rt/` folder to Drive at:**
```
/content/drive/MyDrive/sro_minitaur/output_nl_rt/
    complete/*.all.jsonl
    retest/*.all.jsonl
```
(same layout as the existing `output_nl/`, ~80 MB).

In [ ]:
# setup
from google.colab import drive
drive.mount('/content/drive')
import os
if os.path.isdir('/content/sro-minitaur-transfer'):
    !cd /content/sro-minitaur-transfer && git pull
else:
    !git clone https://github.com/YifeiCAO/sro-minitaur-transfer.git /content/sro-minitaur-transfer
# liger-kernel = fused cross-entropy -> removes the [B,L,V] logits bottleneck -> bigger batch
!pip -q install peft bitsandbytes scikit-learn liger-kernel

RT = '/content/drive/MyDrive/sro_minitaur/output_nl_rt'
assert os.path.isdir(RT + '/complete'), 'upload output_nl_rt/ to Drive first (see cell above)'
print('RT data:', sorted(os.listdir(RT + '/complete'))[:4], '...')

In [ ]:
# 1) train M_pop_RT (8B SFT on RT transcripts; choice + RT both in the loss).
#    A100-40G + Liger fused-CE: batch 12 fills the card (~32-36 GB) and trains faster.
#    Check the log: "[liger] fused kernels enabled" -> batch 12 is safe (push to 16 if you like).
#    If it says "[liger] not active", OR you OOM -> drop --batch-size to 4 (logits-bound).
!cd /content/sro-minitaur-transfer && python scripts/finetune_mpop.py \
    --subset starting_subset \
    --nl-dir /content/drive/MyDrive/sro_minitaur/output_nl_rt \
    --out /content/drive/MyDrive/sro_minitaur/mpop_rt \
    --max-seq-len 4096 --batch-size 12 --grad-accum 2

In [ ]:
# 2) surprise transfer with M_pop_RT on the RT data (choice + RT residual).
#    Batched for A100-40G: --batch-tokens 49152 fills VRAM (8B 4-bit is only ~6 GB).
#    Caches to results/surprise_rt/, writes surprise_matrix_summary_rt.json
!cd /content/sro-minitaur-transfer && python scripts/build_surprise_matrix.py \
    --mpop /content/drive/MyDrive/sro_minitaur/mpop_rt \
    --nl-dir /content/drive/MyDrive/sro_minitaur/output_nl_rt \
    --subset starting_subset --max-seq-len 4096 --batch-tokens 49152 --tag rt

In [ ]:
# 3) compare to the choice-only baselines
import json
r = '/content/drive/MyDrive/sro_minitaur/results'
labels = {'8b_raw': 'choice-only raw 8B', '': 'choice-only M_pop', 'rt': 'RT model (M_pop_RT)'}
for tag in ['8b_raw', '', 'rt']:
    suf = f'_{tag}' if tag else ''
    try:
        s = json.load(open(f'{r}/surprise_matrix_summary{suf}.json'))
        print(f"{labels[tag]:22s}  within={s['within']:.3f}  across={s['across']:.3f}  chance={s['chance']:.3f}")
    except FileNotFoundError:
        print(f'{labels[tag]}: not found')
print('\nwithin(RT) > within(choice-only ~0.14) => RT carries individual signal: thesis confirmed + a better model.')
print('within(RT) ~= choice-only       => binned-RT tokens did not add transferable signal (try Approach B).')

## Approach B — continuous RT regression head (next, if A is promising)

Instead of discretizing RT into tokens, add a small head that reads M_pop's hidden
state at each decision and predicts **log-RT** (regression, or the params of a
log-normal / shifted-lognormal), trained jointly with the choice loss. More faithful
to continuous RT, but needs model surgery + a custom training loop (doesn't reuse
`train_mpop`). The individual signal would then live in the per-trial RT prediction
residual.

This is a separate build — **ask and I'll add `scripts/train_rt_head.py`** once A shows
whether RT moves the needle. (Approach C = DDM-grounded joint (choice, RT) likelihood —
the most principled, biggest build, closest to the RL/DDM cognitive-modeling line.)